In [ ]:
from notebook_setup import setup_notebook_environment

PROJECT_ROOT, SIMULATION_PATH = setup_notebook_environment()

: 

In [ ]:
import os

#define the path to the simulation data
DATA_PATH = os.path.join(SIMULATION_PATH,"data")

#define the path to the fit output directory
OUTPUT_PATH = os.path.join(PROJECT_ROOT,"outputs")

#find the models where simulation data exists
simulation_data_models = [f for f in os.listdir(DATA_PATH) if not f.startswith(".")]



In [ ]:
import glob
import nibabel as nib
from src.utils.most_recent_output_file import most_recent_output_file
import torch

sim_data = {}

for model in simulation_data_models:    
    #get the simulated data, ground truth parameters and mask for each model and store in a dictionary
    sim_data[model + "_data"] = torch.from_numpy(nib.load(glob.glob(os.path.join(DATA_PATH, model, "*_data.nii.gz"))[0]).get_fdata())
    sim_data[model + "_gt_params"] = torch.from_numpy(nib.load(glob.glob(os.path.join(DATA_PATH, model, "*_params.nii.gz"))[0]).get_fdata())
    sim_data[model + "_mask"] = torch.from_numpy(nib.load(glob.glob(os.path.join(DATA_PATH, model, "*_mask.nii.gz"))[0]).get_fdata())
    
    #get the fitted parameters for each model and store in the dictionary
    sim_data[model + "_fitted_params"] = torch.from_numpy(nib.load(most_recent_output_file(OUTPUT_PATH, model)).get_fdata())
    
    
    
    

In [ ]:
#flatten the images

from src.utils.preprocessing import img2voxel

for model in simulation_data_models:
    sim_data[model + "_gt_params_flat"], sim_data[model + "_mask_flat"]= img2voxel(sim_data[model + "_gt_params"], sim_data[model + "_mask"])
    sim_data[model + "_fitted_params_flat"], sim_data[model + "_mask_flat"] = img2voxel(sim_data[model + "_fitted_params"], sim_data[model + "_mask"])

In [ ]:
import matplotlib.pyplot as plt

from src.model_maker import ModelMaker

for model in simulation_data_models:   
    
    modelfunc = ModelMaker(model)
    
    n_maps = sim_data[model + "_gt_params_flat"].shape[1]

    if modelfunc.n_fractions >= 1:
        n_maps = n_maps + 1 #to include the final fraction map
        
    _, ax = plt.subplots(1, n_maps, figsize=(5 * n_maps, 3))
        
   
    
    for i in range(n_maps):
        if i == n_maps-1 and modelfunc.n_fractions >= 1: #if it's the final fraction map
            final_fraction_fitted = 1 - sim_data[model + "_fitted_params_flat"][:,  modelfunc.n_parameters:modelfunc.n_parameters+modelfunc.n_fractions-1].sum(dim=1)
            final_fraction_gt = 1 - sim_data[model + "_gt_params_flat"][:, modelfunc.n_parameters:modelfunc.n_parameters+modelfunc.n_fractions-1].sum(dim=1)
            im = ax[i].plot(final_fraction_gt, final_fraction_fitted, "o", markersize=0.1)
            ax[i].set_title(
                f"f_{modelfunc.n_fractions} "
                f"({modelfunc.compartment_names[modelfunc.n_fractions]})"
            )
        else:
            im = ax[i].plot(sim_data[model + "_gt_params_flat"][:,i], sim_data[model + "_fitted_params_flat"][:,i], "o", markersize=0.1)
        
            ax[i].set_title(
                f"{modelfunc.parameter_names[i]} "
                f"({modelfunc.compartment_names[modelfunc.compartment_indices[i]]})"
            )
            
            ax[i].set_xlabel("Ground Truth") 
            ax[i].set_ylabel("Fitted")
            # ax[i].set_xlim(modelfunc.parameter_ranges[i])
            # ax[i].set_ylim(modelfunc.parameter_ranges[i]) 
        
            
        


